In [1]:
import warnings
warnings.filterwarnings("ignore")

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

from pyspark.sql.types import *
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local-cluster[3,2,2048]")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print(sc.getConf().get("spark.driver.memory", "NOT SET"))

2g


In [2]:
my_schema = """
VendorID INT,
tpep_pickup_datetime TIMESTAMP,
tpep_dropoff_datetime TIMESTAMP,
passenger_count DOUBLE,
trip_distance DOUBLE,
RatecodeID DOUBLE,
store_and_fwd_flag STRING,
PULocationID INT,
DOLocationID INT,
payment_type INT,
fare_amount DOUBLE,
extra DOUBLE,
mta_tax DOUBLE,
tip_amount DOUBLE,
tolls_amount DOUBLE,
improvement_surcharge DOUBLE,
total_amount DOUBLE,
congestion_surcharge DOUBLE,
Airport_fee DOUBLE,
cbd_congestion_fee DOUBLE
"""

In [3]:
from pyspark.sql.functions import *
df = spark.read.format('csv').option('header', 'true').schema(my_schema).load('yellow_tripdata_combined.csv')

group_df = df.groupBy('passenger_count').agg(sum('fare_amount'))

spark.sparkContext.setJobDescription('groupby ---> write data (before salting)')
group_df.write.format('noop').mode('overwrite').save()

In [4]:
spark.conf.set('spark.sql.shuffle.partitions', 15)

In [5]:
SALT_RANGE = 14

df = df.withColumn('salt', (rand()*SALT_RANGE).cast('int'))
df_partial = df.groupBy('passenger_count', 'salt').agg(sum('fare_amount').alias('partial_fare'))
df_final = df_partial.groupBy('passenger_count').agg(sum('partial_fare').alias('total_fare'))

spark.sparkContext.setJobDescription('groupby ---> write data (after salting)')
df_final.write.format('noop').mode('overwrite').save()